<div dir="rtl">

# ⚙️ الجزء 2ب: هندسة الفيديو في الواقع الحقيقي (المستوى المتقدم) 🚀

في هذا المعمل، سننتقل من مجرد كتابة أكواد بسيطة إلى **مستوى الهندسة المعتمد في الصناعة**. 

عند بناء تطبيقات الرؤية الحاسوبية (Computer Vision)، غالباً ما تكون العقبة ليست في نموذج الذكاء الاصطناعي نفسه، بل في كيفية تعامل لغة البرمجة (Python) مع المعالج (CPU).

### 🍳 النموذج الذهني: من المطبخ إلى المصنع
*   **التسلسلي (Sequential)**: طباخ واحد، موقد واحد. يجهز مكوناً واحداً، يطبخه، ثم يجهز المكون التالي. (**حظر العمل - Blocking**)
*   **الخيوط المتعددة (Threaded/Async)**: طباخ واحد، عدة مؤقتات. يجهز المكون، يضعه في الفرن، ويبدأ في تجهيز المكون التالي بينما يعمل المؤقت. (**التزامن - Concurrency**)
*   **المعالجة المتعددة (Multiprocessing)**: عدة طباخين، عدة مطابخ. الجميع يعمل على أطباقهم في نفس الوقت بشكل مستقل. (**التوازي - Parallelism**)

</div>

<div dir="rtl">

## 🛠️ 1. التجهيز وحقيقة استهلاك المعالج (CPU-Bound)
سنقوم بتغيير دالة `heavy_inference` لتكون **كثيفة الاستهلاك للمعالج** (عبر حسابات رياضية معقدة) بدلاً من مجرد الانتظار (`time.sleep`). 
هذا سيكشف نقاط الضعف في "الخيوط" (Threads) و"البرمجة غير المتزامنة" (Async) عند التعامل مع نماذج الذكاء الاصطناعي التي تعتمد بشكل كامل على قدرة المعالج.

</div>

In [1]:
import time
import asyncio
import matplotlib.pyplot as plt
from threading import Thread
from queue import Queue
from multiprocessing import Process, Queue as MPQueue

# استيراد المكتبات الأساسية للتعامل مع الوقت، التزامن، والرسم البياني

<div dir="rtl">

## 🐢 2. النهج الأول: التسلسلي (الأساس القياسي)
هنا نقوم بكل شيء خطوة بخطوة. ننتظر وصول الإطار من الكاميرا، ثم نقوم بمعالجته، ثم ننتظر الإطار التالي.

</div>

In [2]:
def heavy_inference_seq(frame):
    # محاكاة لعملية معالجة ثقيلة (ذكاء اصطناعي)
    count = 0
    for i in range(10**5): 
        count += i
    return frame

def run_sequential(frames_to_process=30):
    start_time = time.time()
    for i in range(frames_to_process):
        time.sleep(0.03) # محاكاة انتظار وصول الإطار من الكاميرا (30 FPS)
        heavy_inference_seq(None) # معالجة الإطار
    duration = time.time() - start_time
    print(f"الوقت الإجمالي للتنفيذ التسلسلي: {duration:.2f} ثانية")
    return duration

seq_time = run_sequential()

<div dir="rtl">

## 🧵 3. النهج الثاني: الخيوط المتعددة (حل الإدخال والإخراج - I/O)
هذا النهج ممتاز عند انتظار الكاميرات أو قواعد البيانات، لكنه لا يزال محدوداً بسبب "قفل المفسر العام" (GIL) في بايثون، مما يمنعه من استغلال كافة أنوية المعالج في العمليات الحسابية.

</div>

In [3]:
class ThreadedProcessor:
    def __init__(self, frames_to_process):
        self.q = Queue(maxsize=1) # طابور لتخزين الإطارات
        self.frames_to_process = frames_to_process
        
    def producer(self):
        # مهمة "المنتج": جلب الإطارات من الكاميرا
        for _ in range(self.frames_to_process):
            time.sleep(0.03)
            if self.q.full():
                try: self.q.get_nowait()
                except: pass
            self.q.put(True)
        self.q.put(None) # إشارة النهاية
            
    def run(self):
        start_time = time.time()
        # بدء خيط (Thread) منفصل لجلب البيانات
        t = Thread(target=self.producer)
        t.start()
        
        while True:
            frame = self.q.get()
            if frame is None: break
            heavy_inference_seq(frame) # معالجة الإطار في الخيط الرئيسي
        t.join()
        duration = time.time() - start_time
        print(f"الوقت الإجمالي للخيوط (Threading): {duration:.2f} ثانية")
        return duration

thread_time = ThreadedProcessor(30).run()

<div dir="rtl">

## 🔄 4. النهج الثالث: البرمجة غير المتزامنة (Async)
نستخدم `asyncio.Queue` لمحاكاة تدفق البيانات في الوقت الحقيقي. هذا النهج رائع جداً للكاميرات المتصلة عبر الشبكة (Web Streams).

</div>

In [4]:
async def async_stream_processor(frames_to_process=30):
    queue = asyncio.Queue(maxsize=1)
    
    async def producer():
        # محاكاة وصول الإطارات بشكل غير متزامن
        for i in range(frames_to_process):
            await asyncio.sleep(0.03) # انتظار وصول الإطار
            if queue.full(): await queue.get()
            await queue.put(i)
        await queue.put(None)

    async def consumer():
        # معالجة الإطارات فور وصولها
        while True:
            frame = await queue.get()
            if frame is None: break
            # محاكاة معالجة غير متزامنة
            await asyncio.sleep(0.05) 
            
    start = time.time()
    await asyncio.gather(producer(), consumer())
    duration = time.time() - start
    print(f"الوقت الإجمالي للـ Async: {duration:.2f} ثانية")
    return duration

async_time = await async_stream_processor()

<div dir="rtl">

## 🏗️ 5. المعيار الصناعي: المعالجة المتعددة الآمنة (Multiprocessing)

تشغيل المعالجة المتعددة (`multiprocessing`) داخل Jupyter Notebook قد يكون أمراً صعباً بسبب كيفية تعامل النظام مع الوحدة الرئيسية (Main Module). في أنظمة Windows و macOS، لا يمكن للمفكرة تصدير دوالك بسهولة إلى العمليات الفرعية.

### الخطوة 1: إنشاء ملف المحرك (Engine File)
سنستخدم خدعة `%%writefile` لإنشاء ملف مساعد على القرص الصلب. بهذه الطريقة، سيكون لدى العملية الفرعية ملف فيزيائي لتقرأ منه الأكواد.

</div>

In [5]:
%%writefile engine.py
import time

def heavy_inference(frame):
    """محاكاة لحمل معالجة حقيقي على المعالج."""
    count = 0
    for i in range(10**5):
        count += i
    return frame

def mp_worker(in_q, out_q):
    """وظيفة العامل التي تعيش في عملية منفصلة تماماً."""
    while True:
        frame = in_q.get()
        if frame is None: 
            break
        result = heavy_inference(frame)
        out_q.put(result)

<div dir="rtl">

### الخطوة 2: مشغل المفكرة (Notebook Runner)
الآن، نقوم باستيراد هذا الملف وتشغيل المنطق البرمجي. هذا النهج يمنع حدوث التوقفات المفاجئة (Deadlocks) عبر استخدام حلقة متكاملة تغذي العامل بالبيانات وتستقبل النتائج باستمرار.

</div>

In [6]:
import engine  # استيراد الملف الذي أنشأناه للتو
from multiprocessing import Process, Queue as MPQueue

def run_multiprocessing_notebook(frames_to_process=30):
    in_q = MPQueue(maxsize=1) # طابور المدخلات
    out_q = MPQueue()        # طابور المخرجات
    
    # بدء العملية الفرعية (Worker Process)
    worker = Process(target=engine.mp_worker, args=(in_q, out_q))
    worker.start()
    
    start_time = time.time()
    sent_count = 0
    processed_count = 0

    print(f"⏳ جاري معالجة {frames_to_process} إطاراً عبر المعالجة المتعددة...")

    while processed_count < frames_to_process:
        # 1. تزويد العامل بالبيانات (المنتج)
        if sent_count < frames_to_process:
            if not in_q.full():
                in_q.put(True)
                sent_count += 1
        elif sent_count == frames_to_process:
            in_q.put(None) # إشارة التوقف للعامل
            sent_count += 1 

        # 2. سحب النتائج (المستهلك)
        while not out_q.empty():
            out_q.get_nowait()
            processed_count += 1
            
        time.sleep(0.01) # راحة بسيطة لمنع استهلاك المعالج بنسبة 100% في هذه الحلقة

    worker.join()
    duration = time.time() - start_time
    print(f"✅ نجاح! الوقت الإجمالي: {duration:.2f} ثانية | السرعة: {frames_to_process/duration:.1f} إطار في الثانية")
    return duration

mp_time = run_multiprocessing_notebook()

<div dir="rtl">

### لماذا ينجح هذا الإصدار؟
1.  **حل الاستيراد**: عبر استيراد `engine` كملف، تعرف العملية الفرعية بالضبط أين تجد دالة العمل.
2.  **الحلقة المتكاملة**: بدلاً من الانتظار المباشر، نقوم بالدوران حتى تنتهي معالجة كافة الإطارات.
3.  **السحب النشط**: التحقق المستمر من `out_q.empty()` يمنع انسداد الأنابيب الداخلية لنقل البيانات.

**💡 نصيحة للمحترفين**: إذا قمت بتعديل ملف `engine.py` بنفسك، يجب عليك **إعادة تشغيل النواة (Restart Kernel)** لتطبيق التغييرات.

</div>

<div dir="rtl">

## 📊 6. المقارنة النهائية والتحليل

| الطريقة | الأفضل لـ | السبب |
| :--- | :--- | :--- |
| **التسلسلي (Sequential)** | الأكواد البسيطة | سهلة التنقيح (Debug) ولكنها بطيئة جداً. |
| **الخيوط (Threads)** | الويب وقواعد البيانات | ممتازة عند "الانتظار" لاستجابة خارجية. |
| **الـ AsyncIO** | آلاف الاتصالات | أقل استهلاك للموارد عند وجود عمليات إدخال/إخراج ضخمة. |
| **المعالجة المتعددة (MP)** | **استنتاج الذكاء الاصطناعي** | الطريقة الوحيدة لاستغلال 100% من قدرات المعالج والرسوميات. |

</div>

In [7]:
results = {
    "Sequential": seq_time,
    "Threaded": thread_time,
    "Async (Stream)": async_time,
    "Multiprocessing": mp_time
}

# رسم بياني للمقارنة
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), [30/t for t in results.values()], color=['red', 'orange', 'blue', 'green'])
plt.ylabel("Effective FPS (Higher is Better)")
plt.title("Real-World Video Processing Throughput")
plt.show()